# Norris Phase 3 revision — reason-then-edit v3 (dev only) and protocol freeze

**What this does (PI instruction, 22 Oct 2026):** simplify the reason-then-edit diagnostic contract to the minimum structure (`diagnosis` → `edited_translation`), rerun the affected **dev** conditions for both models (400 + 400 outputs on the same 100 pilot IDs), check the reconstructed sentence splitter on dev, and build the freeze manifest. **Devtest is never opened in this notebook.**

Runtime: **A100 GPU, High-RAM**. Colab Secrets: `HF_TOKEN` and `OPENAI_API_KEY` with notebook access ON.

**Continuing after 23 Oct:** cells 4–7 are already complete on Drive. Run only **1 → 2 → 8 → 9 → 10 → 11** (cells 4–7 are safe if rerun: saved preparation and records are reused, no new API calls).

Run cells **in order**. If any cell stops, copy the error and send it — do not edit code. Expected time ≈ 60–75 min. GPT cost ≈ $0.05–0.10.

**Predeclared reliability gate (recorded before the rerun):** for each model, fallbacks ≤ 20/400 overall and ≤ 10/100 in every direction. Operational only; COMET is never used to accept/reject. One revision only — if the gate fails we stop and the PI decides.

In [ ]:
# 1 — Mount Drive, clone the Norris repo at REF, load Colab Secrets (never printed).
from pathlib import Path
import os, subprocess, sys, json, hashlib, zipfile
from google.colab import drive, userdata
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/norris')
REPO = Path('/content/csudh-norris-ape-2026')
REF = 'phase3-rte-v3-freeze'
URL = 'https://github.com/SHAZAN01/csudh-norris-ape-2026.git'
import shutil
shutil.rmtree(REPO, ignore_errors=True)  # always a fresh clone: robust to stale folders and rewritten branches
subprocess.run(['git', 'clone', '--branch', REF, URL, str(REPO)], check=True)
COMMIT = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
print('REPO COMMIT:', COMMIT)
assert (REPO / 'src/common/norris_protocol.py').exists(), 'Wrong branch content: push the patched branch first.'
for name in ['HF_TOKEN', 'OPENAI_API_KEY']:
    try:
        os.environ[name] = userdata.get(name) or ''
    except Exception:
        os.environ[name] = ''
    print(name, 'available' if os.environ[name] else 'MISSING (enable it in Colab Secrets if this notebook needs it)')
LOGS = ROOT / 'outputs/logs/colab'
LOGS.mkdir(parents=True, exist_ok=True)

def sh(cmd, log='run.log'):
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd)[:240], flush=True)
    env = os.environ.copy()
    for k in ['PYTHONPATH', 'UV_SYSTEM_PYTHON', 'VIRTUAL_ENV']:
        env.pop(k, None)
    env['TOKENIZERS_PARALLELISM'] = 'false'
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env) as p, \
            open(LOGS / log, 'a', encoding='utf-8') as f:
        for line in p.stdout:
            print(line, end='', flush=True)
            f.write(line)
    if p.returncode:
        raise RuntimeError(f'Stopped (exit {p.returncode}). Copy the error above and send it; log: {LOGS / log}')

sh([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
def venv(path, py):
    if not Path(path, 'bin/python').exists():
        sh([sys.executable, '-m', 'uv', 'venv', '--python', py, path])
    return Path(path, 'bin/python')
def uv(py, *pkgs):
    sh([sys.executable, '-m', 'uv', 'pip', 'install', '--python', py, *pkgs], log='install.log')
print('SETUP OK')

In [ ]:
# 2 — Environments (≈10 min first time)

# Llama/GPT environment (vLLM 0.11.2 exactly as in the measured pilot; openai 1.109.1 for AsyncOpenAI)
LLAMA_PY = venv('/content/env_llama', '3.11')
uv(LLAMA_PY, 'vllm==0.11.2', 'transformers==4.57.1', 'pydantic==2.12.3', 'openai==1.109.1')
sh([LLAMA_PY, '-c', "import torch, vllm, pydantic, openai; from vllm.sampling_params import StructuredOutputsParams; "
    "print('vllm', vllm.__version__, 'torch', torch.__version__, 'openai', openai.__version__); assert torch.cuda.is_available()"])

# Evaluation environment (same COMET stack as the audited pilot evaluation)
EVAL_PY = venv('/content/env_eval', '3.11')
uv(EVAL_PY, 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124')
uv(EVAL_PY, 'torch==2.6.0', 'unbabel-comet==2.2.7', 'transformers==4.49.0', 'huggingface-hub==0.36.0', 'numpy==1.26.4',
   'sacrebleu==2.6.0', 'torchmetrics==0.10.3', 'setuptools==80.9.0', 'langdetect==1.0.9', 'pydantic==2.12.3',
   'rapidfuzz==3.10.1', 'pysbd==0.3.4', 'matplotlib==3.9.2')
sh([EVAL_PY, '-c', "import torch, comet, transformers, numpy, sacrebleu, langdetect, pydantic; "
    "print('torch', torch.__version__, 'transformers', transformers.__version__, 'numpy', numpy.__version__); assert torch.cuda.is_available()"])

In [ ]:
# 3 — Offline tests of the frozen code (no GPU/API use)
sh([LLAMA_PY, REPO / 'tests/frozen/test_protocol.py'])
sh([LLAMA_PY, REPO / 'tests/phase4/test_engine_offline.py'])
sh([EVAL_PY, REPO / 'tests/phase5/test_stats.py'])

In [ ]:
# 4 — Build the 400 dev reason-then-edit v3 jobs. FIRST proves the frozen builder reproduces all
#     1,200 measured v2 job prompts byte-for-byte (so only the reason-then-edit contract changes).
sh([LLAMA_PY, REPO / 'src/phase3/post_editing/prepare_rte_v3_dev.py', '--root', ROOT], log='rte_v3_prepare.log')
JOBS = Path(json.loads((ROOT / 'outputs/phase3_post_editing/rte_v3_latest.json').read_text())['jobs_dir'])
print('JOBS DIR:', JOBS)

In [ ]:
# 5 — GPT-4o Mini: smoke (8 outputs) then full (400). Resumable; never repeats an uncertain paid call.
ENGINE = REPO / 'src/post_editing/run_main_experiments.py'
for stage in ['smoke', 'full']:
    sh([LLAMA_PY, ENGINE, '--jobs-dir', JOBS, '--model', 'gpt4o-mini', '--stage', stage,
        '--max-estimated-usd', '2'], log='rte_v3_gpt.log')

In [ ]:
# 6 — Llama 3 8B Instruct (vLLM): smoke (8) then full (400). ≈10 min incl. model load.
for stage in ['smoke', 'full']:
    sh([LLAMA_PY, ENGINE, '--jobs-dir', JOBS, '--model', 'llama3-8b', '--stage', stage], log='rte_v3_llama.log')

In [ ]:
# 7 — Score v3 (COMET/BLEU/chrF++), re-verify 1,600 stored v2 ZS/OS outcomes under the frozen contract,
#     recheck the 400 pilot baselines, and evaluate the predeclared reliability gate.
sh([EVAL_PY, REPO / 'src/phase3/post_editing/analyze_rte_v3_dev.py', '--root', ROOT], log='rte_v3_analysis.log')
A = Path(json.loads((ROOT / 'outputs/phase3_post_editing/rte_v3_analysis_latest.json').read_text())['output_dir'])
S = json.loads((A / 'rte_v3_summary.json').read_text())
print('\nGATE PASS:', S['gate_pass']); print(json.dumps(S['gate'], indent=1)); print('NEXT:', S['next_step'])

## Stop here if the gate FAILED
Do **not** run cells 8–10. Run cell 11 (evidence ZIP) and send it. The PI decides whether to retain reason-then-edit as an explicit failure condition or remove it.

In [ ]:
# 8 — NLLB environment for the splitter dev gates

# NLLB environment (matches the measured dev baseline run: Python 3.13, torch 2.11.0+cu130, transformers 5.18.0)
NLLB_PY = venv('/content/env_nllb', '3.13')
uv(NLLB_PY, 'torch==2.11.0', '--index-url', 'https://download.pytorch.org/whl/cu130')
uv(NLLB_PY, 'torch==2.11.0', 'transformers==5.18.0', 'tokenizers==0.23.2', 'sentencepiece==0.2.2',
   'huggingface_hub==1.33.0', 'pysbd==0.3.4')
sh([NLLB_PY, '-c', "import sys, torch, transformers; print(sys.version.split()[0], torch.__version__, transformers.__version__, torch.cuda.is_available())"])

In [ ]:
# 9 — S0 baseline audit on DEV ONLY (devtest untouched).
#  (a) splitter must reproduce all 2,991 saved dev segmentations exactly (CPU, seconds)
#  (b) generation audit: exact join rule, frozen fragment variant run twice (must be identical),
#      >=95% exact agreement with the saved dev split translations, dev unsplit regeneration reported (GPU, ~8 min)
GEN = REPO / 'src/baselines/generate_devtest_baseline.py'
sh([NLLB_PY, GEN, '--stage', 'verify-dev-segmentation', '--root', ROOT], log='splitter_gates.log')
sh([NLLB_PY, GEN, '--stage', 'audit-dev-generation', '--root', ROOT], log='splitter_gates.log')

In [ ]:
# 10 — Build and DECLARE the protocol freeze (only possible if the RTE gate and both splitter gates passed).
sh([EVAL_PY, REPO / 'src/freeze/build_freeze_manifest.py', '--root', ROOT, '--git-commit', COMMIT, '--declare'],
   log='freeze.log')
FREEZE = ROOT / 'outputs/frozen/protocol_freeze.json'
print('FREEZE SHA256:', hashlib.sha256(FREEZE.read_bytes()).hexdigest())

In [ ]:
# 11 — Evidence ZIP (download and upload it in the chat). Numeric files are publishable; raw text stays private.
from google.colab import files
from datetime import datetime, timezone
JOBS = Path(json.loads((ROOT / 'outputs/phase3_post_editing/rte_v3_latest.json').read_text())['jobs_dir'])
_ap = ROOT / 'outputs/phase3_post_editing/rte_v3_analysis_latest.json'
if _ap.exists():
    A = Path(json.loads(_ap.read_text())['output_dir'])
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
zp = ROOT / f'outputs/reports/Norris_Phase3_RTEv3_Freeze_Evidence_{stamp}.zip'
zp.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in [JOBS / 'prepared_manifest.json', *JOBS.glob('runs/*/*/run_spec.json'), *JOBS.glob('runs/*/*/run_summary.json'),
              *JOBS.glob('runs/*/*/smoke_summary.json'), *JOBS.glob('runs/*/*/requirements_resolved.txt'),
              *JOBS.glob('generations/*.jsonl'), *JOBS.glob('logs/*.log')]:
        z.write(p, 'rte_v3/' + str(p.relative_to(JOBS)))
    if 'A' in globals():
        for p in A.iterdir():
            if p.is_file():
                z.write(p, 'analysis/' + p.name)
    for p in list((ROOT / 'outputs/phase4').glob('gate_*.json')) + [ROOT / 'outputs/frozen/protocol_freeze.json']:
        if p.exists() and 'PRIVATE' not in p.name:
            z.write(p, 'gates_and_freeze/' + p.name)
    for p in LOGS.glob('*.log'):
        z.write(p, 'colab_logs/' + p.name)
print(zp, round(zp.stat().st_size / 1e6, 1), 'MB')
files.download(str(zp))